In [ ]:
pip install roboflow opencv-python numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 121.6 MB/s  0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.3
    Uninstalling typer-0.27.3:
      Successfully uninstalled typer-0.27.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [roboflow]


In [ ]:
from roboflow import Roboflow


rf = Roboflow(api_key="Uhaqiw1Iwi8RGRmDdMMR")
project = rf.workspace("lynnho-ingaga-s-workspace").project("steel-pipe-jjjvt")
version = project.version(1)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Steel-Pipe-1 in yolov8:: 100%|██████████| 42792/42792 [00:09<00:00, 4569.05it/s] 


In [ ]:
import yaml
import cv2
import numpy as np
import glob
import os
import random

# 1. Lecture et affichage des 6 classes du dataset
with open("Steel-Pipe-1/data.yaml", "r") as f:
    data_cfg = yaml.safe_load(f)
print("Classes du dataset :", data_cfg["names"])

Classes du dataset : ['Deformation', 'Desposition', 'Disconnet', 'Misalignment', 'Obstacle', 'Rupture']


In [ ]:
def generate_pipe_crawler_robot_video(
    test_images_dir="Steel-Pipe-1/test/images",
    output_video="sim_robot_crawler_pipe.mp4",
    num_images=25,
    fps=30,
    out_width=1280,
    out_height=720,
    frames_per_segment=75,  # 2.5 secondes d'avancée par section de tuyau
    seed=42
):
    random.seed(seed)
    all_images = glob.glob(os.path.join(test_images_dir, "*.jpg")) + \
                 glob.glob(os.path.join(test_images_dir, "*.png"))

    selected_paths = random.sample(all_images, min(num_images, len(all_images)))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_video, fourcc, fps, (out_width, out_height))

    total_frames = len(selected_paths) * frames_per_segment
    print(f"Génération de {total_frames} frames ({total_frames/fps:.1f}s) - Mode Robot Chenillé...")

    global_frame = 0
    distance_meters = 0.0

    for idx, path in enumerate(selected_paths):
        img = cv2.imread(path)
        if img is None:
            continue
        img = cv2.resize(img, (out_width, out_height))

        for f in range(frames_per_segment):
            progress = f / frames_per_segment  # Va de 0.0 à 1.0

            # 1. Effet d'avancée dans le tuyau (zoom progressif de 1.0x à 1.35x)
            zoom = 1.0 + 0.35 * progress
            new_w = int(out_width * zoom)
            new_h = int(out_height * zoom)
            zoomed = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

            # 2. Simulation des secousses de chenilles du robot (tangage vertical + léger roulis)
            bump_y = int(np.sin(global_frame * 0.6) * 4 + random.uniform(-1.5, 1.5))
            sway_x = int(np.cos(global_frame * 0.25) * 3)

            cx = (new_w - out_width) // 2 + sway_x
            cy = (new_h - out_height) // 2 + bump_y

            cx = max(0, min(cx, new_w - out_width))
            cy = max(0, min(cy, new_h - out_height))

            frame = zoomed[cy:cy + out_height, cx:cx + out_width].copy()

            # 3. Effet phare LED du robot (légère variation d'éclairage central)
            light = 0.95 + 0.08 * np.sin(global_frame * 0.05)
            frame = np.clip(frame * light, 0, 255).astype(np.uint8)

            # 4. HUD réaliste de robot d'inspection de canalisation (Crawler CCTV)
            distance_meters += 0.02  # Avance à ~0.6 m/s (vitesse standard robot tuyau)
            pitch = np.sin(global_frame * 0.3) * 1.8
            hud_top = f"CRAWLER-BOT-01 | DIST: {distance_meters:06.2f}m | SPD: 0.60m/s | PITCH: {pitch:+.1f} deg"
            hud_bot = f"CAM: FRONT-LED 100% | SEC: {idx+1:02d}/{len(selected_paths)} | FRM: {global_frame:04d}"

            cv2.putText(frame, hud_top, (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
            cv2.putText(frame, hud_bot, (20, out_height - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)

            writer.write(frame)
            global_frame += 1

    writer.release()
    print(f"Terminé ! Vidéo robot générée : {output_video}")

generate_pipe_crawler_robot_video()

Génération de 1875 frames (62.5s) - Mode Robot Chenillé...
Terminé ! Vidéo robot générée : sim_robot_crawler_pipe.mp4


In [ ]:
import glob
import yaml
from collections import Counter

# 1. Lecture du fichier de configuration du dataset généré par Roboflow
# Il contient le nombre de classes (nc), leurs noms (names) et les chemins vers train/val/test
with open("Steel-Pipe-1/data.yaml", "r") as f:
    data_cfg = yaml.safe_load(f)

print("Configuration data.yaml :", data_cfg)
names = data_cfg["names"]

# 2. Parcours de tous les fichiers d'annotations (.txt) du dossier d'entraînement
# Au format YOLO, chaque ligne d'un fichier .txt représente un objet sous la forme :
# <id_classe> <x_centre> <y_centre> <largeur> <hauteur> (valeurs normalisées entre 0 et 1)
label_files = glob.glob("Steel-Pipe-1/train/labels/*.txt")

class_counts = Counter()
empty_images = 0
box_areas = []

for file_path in label_files:
    with open(file_path, "r") as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    # Si le fichier est vide, c'est une "Background image" (tuyau sain sans défaut)
    # C'est très utile pour apprendre au modèle à ne pas générer de faux positifs !
    if len(lines) == 0:
        empty_images += 1
        continue

    for line in lines:
        parts = line.split()
        cls_id = int(parts[0])
        w, h = float(parts[3]), float(parts[4])

        class_counts[names[cls_id]] += 1
        box_areas.append(w * h)  # Surface relative de la boîte dans l'image

print(f"\n--- RÉSULTATS DE L'AUDIT ({len(label_files)} images d'entraînement) ---")
print(f"Images saines sans défaut (Background) : {empty_images} ({empty_images/len(label_files)*100:.1f}%)")
print("\nNombre d'exemples par classe :")
for cls_name, count in class_counts.most_common():
    print(f" - {cls_name:<15} : {count:>5} instances")

avg_area = (sum(box_areas) / len(box_areas)) * 100 if box_areas else 0
print(f"\nTaille moyenne d'une anomalie : {avg_area:.2f}% de la surface de l'image")


Configuration data.yaml : {'names': ['Deformation', 'Desposition', 'Disconnet', 'Misalignment', 'Obstacle', 'Rupture'], 'nc': 6, 'roboflow': {'license': 'CC BY 4.0', 'project': 'steel-pipe-jjjvt', 'url': 'https://app.roboflow.com/lynnho-ingaga-s-workspace/steel-pipe-jjjvt/1', 'version': 1, 'workspace': 'lynnho-ingaga-s-workspace'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}

--- RÉSULTATS DE L'AUDIT (16046 images d'entraînement) ---
Images saines sans défaut (Background) : 0 (0.0%)

Nombre d'exemples par classe :
 - Obstacle        :  4703 instances
 - Rupture         :  4278 instances
 - Deformation     :  4029 instances
 - Desposition     :  3471 instances
 - Misalignment    :  2267 instances
 - Disconnet       :  2092 instances

Taille moyenne d'une anomalie : 22.83% de la surface de l'image


In [ ]:
!pip install -q ultralytics

In [ ]:
import torch
from google.colab import drive
from ultralytics import YOLO

# 2. Vérification de la présence du GPU (indispensable pour 16 000 images)
if not torch.cuda.is_available():
    raise RuntimeError("ATTENTION : Aucun GPU détecté ! Allez dans le menu Colab : Exécution > Modifier le type d'exécution > T4 GPU.")
print(f"GPU prêt : {torch.cuda.get_device_name(0)}")

GPU prêt : Tesla T4


In [ ]:
# 1. Montage forcé de Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. VERROU DE SÉCURITÉ : vérifie que votre dossier existant 'Colab Notebooks' est bien accessible
assert os.path.exists("/content/drive/MyDrive/Colab Notebooks"), "ERREUR : Google Drive n'est pas réellement monté !"

# 3. Création du dossier et du fichier témoin visible immédiatement sur ://google.com
save_root = "/content/drive/MyDrive/Pipe_Inspection_MLE"
os.makedirs(save_root, exist_ok=True)
with open(f"{save_root}/TEST_CONNEXION_OK.txt", "w") as f:
    f.write("Connexion Google Drive confirmée !")
print("OK : Allez vérifier sur ://google.com que TEST_CONNEXION_OK.txt est visible !")

Mounted at /content/drive
OK : Allez vérifier sur ://google.com que TEST_CONNEXION_OK.txt est visible !


In [ ]:
#creation de l e,trainement


In [ ]:
model =YOLO("yolo11s.pt")

In [ ]:
#lancement entrainement


In [ ]:
results=model.train(
    data="Steel-Pipe-1/data.yaml",
    epochs=40,
    patience=8,
    imgsz=640,
    batch=32,
    cache="disk",
    workers=8,
    #data augmentation applique robot
    degrees=15.00,
    flipud=0.0,
    fliplr=0.5,                      # Activé (50%) : un défaut à gauche peut aussi apparaître à droite
    hsv_v=0.4,                       # Simule les variations d'intensité du phare LED du robot
    mosaic=1.0 ,      # Combine 4 images pour apprendre au modèle à détecter plusieurs défauts proches
# --- Sauvegarde sécurisée sur Google Drive ---
    project="/content/drive/MyDrive/Pipe_Inspection_MLE",
    name="yolo11s_6classes_v2",
    save=True,
    plots=True                       # Génère la matrice de confusion et les courbes Precision/Recall
)



Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Steel-Pipe-1/data.yaml, degrees=15.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11s_6classes_v2, nbs=64, nms=None, opset=None, optimize=False,

In [ ]:
#entrainement des 20 autres epocks continuer sur kaggle du fait de l'expiration du GPU gratuit de collab